# AI-Powered Data Quality Monitoring Platform

## Phase 1: Data Ingestion

- Load NYC Taxi trip dataset into Databricks
- Understand the dataset schema
- Explore basic dataset information
- Prepare the data for validation and quality checks

In [0]:
# import required libraries

from pyspark.sql import SparkSession

## Step 1: Load NYC Taxi Dataset from Bronze Layer

The raw parquet dataset has been uploaded into the Bronze layer volume.

Dataset Path:
`/Volumes/taxi_quality_platform/bronze/raw_taxi_data/yellow_tripdata_2024-01.parquet`

We are now loading the dataset using PySpark.

In [0]:
# Load NYC taxi dataset from volume

df = spark.read.parquet(
    "/Volumes/taxi_quality_platform/bronze/raw_taxi_data/yellow_tripdata_2024-01.parquet"
)

## Step 2: Display Sample Records

We are displaying the first 5 records to understand:
- available columns
- sample values
- data structure

In [0]:
# display first 5 rows
df.show(5)

## Step 3: Analyze Dataset Schema

Schema analysis helps us understand:
- column names
- data types
- nullable fields

This is important before performing data quality validations.

In [0]:
df.printSchema()

## Step 4: Count Total Records

This helps us understand the dataset scale.

In [0]:
# count total records 
total_records = df.count()
print(f"Total records: {total_records}")

## Step 5: Display Important Columns

These columns will later be used for:
- null validation
- anomaly detection
- duplicate checks
- quality scoring

In [0]:
# display important columns 
df.select(
    "VendorID",
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "payment_type"
).show(5)

## Step 6: Check Dataset Summary Statistics

Summary statistics help identify:
- unusual values
- negative values
- outliers
- abnormal distributions

These insights are useful for designing validation rules.

In [0]:
# Generate statistics summary

df.select(
    "trip_distance",
    "fare_amount",
    "passenger_count"
).describe().show()

## Step 7: Identify Potential Data Quality Issues

In large-scale enterprise datasets, common quality issues include:

- null values
- duplicate records
- negative fares
- invalid trip distances
- missing passenger counts

In [0]:
negative_fare_df = df.filter(df.fare_amount < 0)
print("Negative Fare Records:", negative_fare_df.count())

In [0]:
# Check for the records with Invalid trip distance

invalid_trip_distance = df.filter(df.trip_distance <= 0)
print(f"Invalid trips: {invalid_trip_distance.count()}")

## Step 9: Explore Null Values in Important Columns

Null values are one of the most common data quality issues in enterprise pipelines.

We will analyze null counts for critical business columns.

In [0]:
from pyspark.sql.functions import col, when, count, sum

In [0]:
# Count null values in important columns 

columns = ["passenger_count", "trip_distance", "fare_amount", "payment_type"]

df.select(
    [
        sum(col(c).isNull().cast("int")).alias(c)
        for c in columns
    ]
).show()

In [0]:
# count zero or negative values in the columns 

columns = ["passenger_count", "trip_distance", "fare_amount", "payment_type"]

df.select(
    [
        sum((col(c) <= 0 ).cast("int")).alias(c)
        for c in columns
    ]
).show()

# Ingestion Phase Summary

In this notebook, we:
- loaded the NYC Taxi dataset into the Bronze layer
- explored schema and business columns
- performed initial data profiling
- identified potential quality issues and anomalies

The raw dataset is now ready for validation processing.